In [19]:
# ============================================================
# Step 1: Verify the Python Environment and Required Packages
# ============================================================
# Purpose:
# 1. Confirm that the Python environment is working correctly.
# 2. Confirm that pandas is available for data processing.
# 3. Confirm that pyodbc is available for connecting to CGMS SQL Server.

# Import pandas for data loading, cleaning, and analysis
import pandas as pd



# Display package versions to confirm the environment setup
print("Python environment is ready!")
print("Pandas version:", pd.__version__)


Python environment is ready!
Pandas version: 2.3.3


In [20]:
# ============================================================
# Step 2: Install the SQL Server Connection Package
# ============================================================
# pyodbc is required to connect Python to the CGMS SQL Server database.
# %pip installs the package into the Python environment used by this notebook.

%pip install pyodbc

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
# ============================================================
# Step 3: Verify the Database Connection Package
# ============================================================
# Import pyodbc and confirm that the installation was successful.

import pyodbc

print("pyodbc is ready!")
print("pyodbc version:", pyodbc.version)

pyodbc is ready!
pyodbc version: 5.3.0


In [22]:
# ============================================================
# Step 4: Check Available SQL Server ODBC Drivers
# ============================================================
# Purpose:
# Check which ODBC drivers are installed on the local computer.
# Python will use one of these drivers to connect to the CGMS SQL Server.

import pyodbc

drivers = pyodbc.drivers()

print("Available ODBC drivers:")
for driver in drivers:
    print("-", driver)

Available ODBC drivers:
- SQL Server


In [23]:
# ============================================================
# Step 5: Connect to the CGMS SQL Server Database
# ============================================================
# Purpose:
# 1. Connect Python to the CGMS_MDL local replica.
# 2. Use Windows Authentication / SSO instead of username and password.
# 3. Verify that the database connection works successfully.

import pyodbc

# Database connection parameters
server = "he-db.mdl.ccm.conti.de"
database = "CGMS_MDL"

# Build the connection string using Windows Authentication
# No username or password is required because SSO is enabled.
connection_string = (
    "DRIVER={SQL Server};"
    f"SERVER={server},1433;"
    f"DATABASE={database};"
    "Trusted_Connection=yes;"
)

# Try to connect to the CGMS SQL Server
try:
    conn = pyodbc.connect(connection_string)

    print("Database connection successful!")
    print("Server:", server)
    print("Database:", database)
    print("Authentication: Windows Authentication / SSO")

except pyodbc.Error as error:
    print("Database connection failed.")
    print("Error:", error)

Database connection successful!
Server: he-db.mdl.ccm.conti.de
Database: CGMS_MDL
Authentication: Windows Authentication / SSO


In [24]:
# ============================================================
# Step 6: Test Data Extraction from CGMS
# ============================================================
# Purpose:
# 1. Verify that Python can execute SQL queries on CGMS_MDL.
# 2. Load a small sample of production order data into pandas.
# 3. Confirm that the database connection and data extraction work correctly.

# Test query: retrieve a small sample from the Orders table
test_query = """
SELECT TOP 10 *
FROM [Orders]
"""

# Execute the SQL query and load the result into a pandas DataFrame
df_test = pd.read_sql(test_query, conn)

# Display the first rows
print("Data extraction successful!")
print("Number of rows:", len(df_test))
print("Number of columns:", len(df_test.columns))

df_test.head()

Data extraction successful!
Number of rows: 10
Number of columns: 73


C:\Users\Wangw3\AppData\Local\Temp\ipykernel_27912\1142827995.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_test = pd.read_sql(test_query, conn)


,OrderID,OrderIDLinked,OrderIDParent,OrderTypeID,WorkCenterID,RecipeID,MaterialMasterID,CarrierTypeID,OrderStatusID,PlantCode,...,OrderIDPaired,ExtTerminated,OrderIDCoupled,WorkCenterNameCoupled,MaterialMasterIDCoupled,OrderIDLeading,WorkCenterNameLeading,MaterialMasterIDLeading,PhaseInOut,IsGAPOrder
0,0100305432,None,None,None,651,119926,992540000200,None,12,9200,...,None,False,None,None,None,None,None,None,,False
1,0100305433,None,None,None,651,119926,992540000200,None,12,9200,...,None,False,None,None,None,None,None,None,,False
2,0100305435,None,None,None,651,119926,992540000200,None,12,9200,...,None,False,None,None,None,None,None,None,,False
3,0100305438,None,None,None,651,119926,992540000200,None,11,9200,...,None,False,None,None,None,None,None,None,,False
4,0100305480,None,None,None,651,119928,992540001700,None,12,9200,...,None,False,None,None,None,None,None,None,,False


In [25]:
# ============================================================
# Step 7: Extract Production Data for TB Capacity Analysis
# ============================================================
# Purpose:
# 1. Extract TB production order data from CGMS_MDL.
# 2. Combine production orders with Work Center, Production Version,
#    and Recipe information.
# 3. Load the SQL query result into a pandas DataFrame.
# 4. Create the main dataset for the following capacity analysis.

# Paste the validated SQL query from DBeaver between the triple quotes
query = """

SELECT

    a.[OrderID],
    a.[OrderIDLinked],
    a.[WorkCenterID],
    a.[RecipeID],
    a.[MaterialMasterID],
    a.[Comment],
    a.[QuantitySet],
    a.[QuantityCorrectionSet],
    a.[QuantityRemaining],
    a.[QuantityDone],
    a.[ShiftNumber],
    a.[ShiftDate],
    a.[ActualStartTime],
    a.[ActualEndTime],
    a.[PlannedStartTime],
    a.[PlannedEndTime],
    a.[ProdVersion],
    a.[ProdVariant],
    a.[DestinationInfo],
    a.[ETOFlag],
    a.[PLCOrderNumber],
    a.[StatusUpdatedOn],
    a.[IsParent],
    a.[CreatedByCGMS],
    a.[InsertedOn],
    a.[UpdatedOn],
    a.[UserIDInsertedBy],
    a.[ExtTerminated],
    a.[OrderIDCoupled],
    a.[WorkCenterNameCoupled],
    a.[MaterialMasterIDCoupled],
    a.[PhaseInOut],
    a.[IsGAPOrder],

    -- Actual production duration in seconds
    DATEDIFF_BIG(
        SECOND,
        a.[ActualStartTime],
        a.[ActualEndTime]
    ) AS ActualProcessingTime,

    -- Actual processing time per finished tire
    CASE
        WHEN a.[QuantityDone] > 0
        THEN
            CAST(
                DATEDIFF_BIG(
                    SECOND,
                    a.[ActualStartTime],
                    a.[ActualEndTime]
                ) AS DECIMAL(18, 4)
            )
            / CAST(a.[QuantityDone] AS DECIMAL(18, 4))
        ELSE NULL
    END AS ActualTimePerTire,

    -- Descriptive fields from related tables
    b.[GlobalName] AS OrderStatus,
    c.[Name] AS WorkCenterName,

    -- IE standard data from ProductionVersion
    d.[CycleTimePieces] AS CycleTimePieces,
    d.[CycleTime] AS IEStandardCycleTime,
    d.[CycleTimeTolerance] AS CycleTimeTolerance,
    d.[TotalTime] AS IEStandardTotalTime,
    d.[StandardSetupTime] AS StandardSetupTime,
    --recipe related description &Prod variant
    r.[Description] AS RecipeDescription,
    r.[ProdVariant] AS ProdVariant

FROM [Orders] a WITH (NOLOCK)

INNER JOIN [OrderStatus] b WITH (NOLOCK)
    ON a.[OrderStatusID] = b.[OrderStatusID]

INNER JOIN [WorkCenter] c WITH (NOLOCK)
    ON a.[WorkCenterID] = c.[WorkCenterID]

LEFT JOIN [ProductionVersion] d WITH (NOLOCK)
    ON a.[MaterialMasterID] = d.[MaterialMasterID]
    AND a.[WorkCenterID] = d.[WorkCenterID]
    
LEFT JOIN [Recipe] r WITH (NOLOCK)
    ON a.[MaterialMasterID] = r.[MaterialMasterID]
    AND a.[RecipeID]  = r.RecipeID 

WHERE c.[Name] LIKE 'TB%'
  AND a.[ActualStartTime] IS NOT NULL
  AND a.[ActualEndTime] IS NOT NULL
  AND a.[ActualEndTime] >= a.[ActualStartTime]
  AND a.[OrderID] NOT LIKE 'CWM%'

ORDER BY a.[InsertedOn] DESC


"""

# Execute the SQL query and load the result into a pandas DataFrame
df_capacity = pd.read_sql(query, conn)

# Check the extracted dataset
print("Data extraction completed!")
print("Number of rows:", len(df_capacity))
print("Number of columns:", len(df_capacity.columns))

# Preview the first 5 records
df_capacity.head()

C:\Users\Wangw3\AppData\Local\Temp\ipykernel_27912\324917190.py:114: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_capacity = pd.read_sql(query, conn)


Data extraction completed!
Number of rows: 47655
Number of columns: 44


,OrderID,OrderIDLinked,WorkCenterID,RecipeID,MaterialMasterID,Comment,QuantitySet,QuantityCorrectionSet,QuantityRemaining,QuantityDone,...,ActualTimePerTire,OrderStatus,WorkCenterName,CycleTimePieces,IEStandardCycleTime,CycleTimeTolerance,IEStandardTotalTime,StandardSetupTime,RecipeDescription,ProdVariant
0,0121654189,None,475,69448,992670434500,None,0.0,0.0,0.0,NaN,...,NaN,Completed,TB122,100.0,4319.0,715.0,71.509003,1006.0,215/60 R17 (C+0),C1
1,0117746701,None,495,148468,992670593700,None,0.0,0.0,0.0,NaN,...,NaN,Completed,TB172,100.0,3959.0,600.0,78.967003,1006.0,195/60 R15 (1+0),C1
2,0117934602,0117934466,495,166319,992670629200,None,163.0,NaN,0.0,163.0,...,59.447853,Completed,TB172,100.0,3959.0,600.0,78.967003,1006.0,195/55 R15 (C+0),C1
3,0117934177,0117934138,495,148468,992670593700,None,110.0,NaN,-11.0,121.0,...,63.322314,Completed,TB172,100.0,3959.0,600.0,78.967003,1006.0,195/60 R15 (1+0),C1
4,0117934477,0117934464,495,166319,992670629200,None,160.0,NaN,-8.0,168.0,...,62.000000,Completed,TB172,100.0,3959.0,600.0,78.967003,1006.0,195/55 R15 (C+0),C1


In [26]:
# ============================================================
# Step 8: Initial Data Quality Check
# ============================================================
# Purpose:
# 1. Check the overall structure of the extracted dataset.
# 2. Review column names and data types.
# 3. Identify missing values in the key fields.
# 4. Prepare the dataset for TB capacity analysis.

# Display the dataset dimensions
print("Dataset shape:", df_capacity.shape)

# Display all column names
print("\nColumns:")
for column in df_capacity.columns:
    print("-", column)

# Check data types
print("\nData types:")
print(df_capacity.dtypes)

# Define key fields for the capacity analysis
key_columns = [
    "OrderID",
    "WorkCenterID",
    "WorkCenterName",
    "MaterialMasterID",
    "RecipeID",
    "QuantityDone",
    "ActualStartTime",
    "ActualEndTime",
    "ActualProcessingTime",
    "ActualTimePerTire",
    "IEStandardCycleTime",
    "StandardSetupTime",
    "RecipeDescription",
    "ProdVariant"
]

# Check missing values in key fields
print("\nMissing values in key analysis fields:")
print(df_capacity[key_columns].isnull().sum())

Dataset shape: (47655, 44)

Columns:
- OrderID
- OrderIDLinked
- WorkCenterID
- RecipeID
- MaterialMasterID
- Comment
- QuantitySet
- QuantityCorrectionSet
- QuantityRemaining
- QuantityDone
- ShiftNumber
- ShiftDate
- ActualStartTime
- ActualEndTime
- PlannedStartTime
- PlannedEndTime
- ProdVersion
- ProdVariant
- DestinationInfo
- ETOFlag
- PLCOrderNumber
- StatusUpdatedOn
- IsParent
- CreatedByCGMS
- InsertedOn
- UpdatedOn
- UserIDInsertedBy
- ExtTerminated
- OrderIDCoupled
- WorkCenterNameCoupled
- MaterialMasterIDCoupled
- PhaseInOut
- IsGAPOrder
- ActualProcessingTime
- ActualTimePerTire
- OrderStatus
- WorkCenterName
- CycleTimePieces
- IEStandardCycleTime
- CycleTimeTolerance
- IEStandardTotalTime
- StandardSetupTime
- RecipeDescription
- ProdVariant

Data types:
OrderID                            object
OrderIDLinked                      object
WorkCenterID                        int64
RecipeID                            int64
MaterialMasterID                   object
Comment 

In [27]:
# ============================================================
# Step 9: Clean and Prepare Data for TB Capacity Analysis
# ============================================================
# Purpose:
# 1. Create a clean working dataset without modifying the raw data.
# 2. Convert key production fields into appropriate data types.
# 3. Remove records that cannot be used for actual capacity calculation.
# 4. Create basic capacity-related features for further analysis.

# Create a separate working DataFrame
# Keep df_capacity unchanged as the original extracted dataset
df_clean = df_capacity.copy()

print("Original dataset:")
print("Rows:", len(df_clean))


# ------------------------------------------------------------
# 9.1 Convert Time Columns to Datetime Format
# ------------------------------------------------------------

time_columns = [
    "ActualStartTime",
    "ActualEndTime",
    "PlannedStartTime",
    "PlannedEndTime"
]

for column in time_columns:
    if column in df_clean.columns:
        df_clean[column] = pd.to_datetime(
            df_clean[column],
            errors="coerce"
        )


# ------------------------------------------------------------
# 9.2 Convert Key Production Fields to Numeric Format
# ------------------------------------------------------------

numeric_columns = [
    "QuantityDone",
    "ActualProcessingTime",
    "ActualTimePerTire",
    "IEStandardCycleTime",
    "StandardSetupTime"
]

for column in numeric_columns:
    if column in df_clean.columns:
        df_clean[column] = pd.to_numeric(
            df_clean[column],
            errors="coerce"
        )


# ------------------------------------------------------------
# 9.3 Remove Records Without Valid Production Quantity
# ------------------------------------------------------------
# QuantityDone must be greater than 0 because actual capacity
# cannot be calculated for orders without finished tires.

df_clean = df_clean[
    df_clean["QuantityDone"].notna() &
    (df_clean["QuantityDone"] > 0)
].copy()


# ------------------------------------------------------------
# 9.4 Remove Records Without Valid Actual Production Time
# ------------------------------------------------------------
# Actual processing time must be greater than 0.

df_clean = df_clean[
    df_clean["ActualProcessingTime"].notna() &
    (df_clean["ActualProcessingTime"] > 0)
].copy()


# ------------------------------------------------------------
# 9.5 Calculate Actual Production Rate
# ------------------------------------------------------------
# ActualCapacityPerHour represents the number of finished tires
# produced per hour during the recorded production period.

df_clean["ActualCapacityPerHour"] = (
    df_clean["QuantityDone"]
    / df_clean["ActualProcessingTime"]
    * 3600
)


# ------------------------------------------------------------
# 9.6 Check the Cleaned Dataset
# ------------------------------------------------------------

print("\nCleaned dataset:")
print("Rows:", len(df_clean))

print("\nRows removed during basic cleaning:")
print(len(df_capacity) - len(df_clean))

print("\nKey capacity indicators:")
print(
    df_clean[
        [
            "QuantityDone",
            "ActualProcessingTime",
            "ActualTimePerTire",
            "ActualCapacityPerHour"
        ]
    ].describe()
)

Original dataset:
Rows: 47655

Cleaned dataset:
Rows: 47528

Rows removed during basic cleaning:
127

Key capacity indicators:
       QuantityDone  ActualProcessingTime  ActualTimePerTire  \
count  47528.000000          47528.000000       47528.000000   
mean     141.204806          11367.369782         120.346628   
std       81.822514           9110.464295        1043.949075   
min        1.000000            334.000000          47.005556   
25%       95.000000           7663.750000          68.936888   
50%      118.000000           9712.000000          77.090909   
75%      161.000000          12971.000000          90.278147   
max      744.000000         582942.000000      111629.000000   

       ActualCapacityPerHour  
count           47528.000000  
mean               45.180302  
std                10.589101  
min                 0.032250  
25%                39.876760  
50%                46.698113  
75%                52.221678  
max                76.586692  


In [28]:
# ============================================================
# Step 10: Investigate Outliers in Actual Production Performance
# ============================================================
# Purpose:
# 1. Identify unusually high actual processing time per tire.
# 2. Compare abnormal orders with normal production performance.
# 3. Check whether extreme values are related to specific machines,
#    materials, recipes, or very small order quantities.
# 4. Keep all records at this stage. No outliers are removed yet.


# ------------------------------------------------------------
# 10.1 Calculate Percentiles for Actual Time per Tire
# ------------------------------------------------------------

percentiles = df_clean["ActualTimePerTire"].quantile(
    [0.50, 0.75, 0.90, 0.95, 0.99, 0.995, 1.00]
)

print("Actual Time per Tire Percentiles (seconds):")
print(percentiles)


# ------------------------------------------------------------
# 10.2 Define a Temporary Outlier Threshold
# ------------------------------------------------------------
# Use the 99th percentile only as an investigation threshold.
# This threshold is NOT used to permanently remove data.

outlier_threshold = df_clean["ActualTimePerTire"].quantile(0.99)

print("\n99th percentile threshold:")
print(round(outlier_threshold, 2), "seconds/tire")


# ------------------------------------------------------------
# 10.3 Identify Outlier Production Orders
# ------------------------------------------------------------

df_outliers = df_clean[
    df_clean["ActualTimePerTire"] > outlier_threshold
].copy()

print("\nNumber of investigated outlier orders:", len(df_outliers))
print(
    "Percentage of dataset:",
    round(len(df_outliers) / len(df_clean) * 100, 2),
    "%"
)


# ------------------------------------------------------------
# 10.4 Review the Most Extreme Production Orders
# ------------------------------------------------------------

outlier_review_columns = [
    "OrderID",
    "WorkCenterName",
    "MaterialMasterID",
    "RecipeID",
    "ProdVariant",
    "QuantityDone",
    "ActualStartTime",
    "ActualEndTime",
    "ActualProcessingTime",
    "ActualTimePerTire",
    "ActualCapacityPerHour",
    "IEStandardCycleTime",
    "StandardSetupTime"
]

df_outlier_review = (
    df_outliers[outlier_review_columns]
    .sort_values(
        by="ActualTimePerTire",
        ascending=False
    )
)

# Display the 30 most extreme orders
df_outlier_review.head(30)

Actual Time per Tire Percentiles (seconds):
0.500        77.090909
0.750        90.278147
0.900       115.794717
0.950       142.464333
0.990       305.261867
0.995       477.538269
1.000    111629.000000
Name: ActualTimePerTire, dtype: float64

99th percentile threshold:
305.26 seconds/tire

Number of investigated outlier orders: 476
Percentage of dataset: 1.0 %


,OrderID,WorkCenterName,MaterialMasterID,RecipeID,ProdVariant,ProdVariant,QuantityDone,ActualStartTime,ActualEndTime,ActualProcessingTime,ActualTimePerTire,ActualCapacityPerHour,IEStandardCycleTime,StandardSetupTime
3628,0126484670,TB2A6,992681045900,175350,G1,G1,1.0,2026-10-20 00:53:08.890,2026-10-21 07:53:37.747,111629,111629.000000,0.032250,5408.0,1006.0
14948,0121354609,TB176,992670332300,45904,C1,C1,9.0,2026-09-27 13:57:35.457,2026-10-04 07:53:17.057,582942,64771.333333,0.055580,7489.0,1006.0
14430,0121976285,TB2C3,992681049600,174574,G1,G1,5.0,2026-09-30 05:47:06.317,2026-10-03 20:20:58.957,311632,62326.400000,0.057760,6336.0,1006.0
14542,0121654442,TB132,992670489200,87997,C1,C1,1.0,2026-10-03 02:33:05.550,2026-10-03 17:43:53.567,54648,54648.000000,0.065876,4319.0,1006.0
14551,0121656789,TB134,992670446900,148298,C1,C1,2.0,2026-10-03 01:36:15.550,2026-10-03 23:33:37.330,79042,39521.000000,0.091091,4675.0,1006.0
14548,0121657069,TB234,992680915400,133908,G1,G1,2.0,2026-10-03 02:11:31.017,2026-10-03 23:33:23.053,76912,38456.000000,0.093613,4675.0,1006.0
14530,0121655741,TB161,992670590000,147315,C1,C1,2.0,2026-10-03 01:22:28.597,2026-10-03 21:30:39.667,72491,36245.500000,0.099323,4533.0,1006.0
14562,0121656809,TB152,992670517200,165531,C1,C1,2.0,2026-10-03 05:18:03.620,2026-10-03 21:08:04.577,57001,28500.500000,0.126314,5297.0,1006.0
14503,0121657136,TB252,992680936600,165527,G1,G1,2.0,2026-10-03 05:27:15.207,2026-10-03 21:13:40.110,56785,28392.500000,0.126794,5297.0,1006.0
14539,0121657086,TB182,992670606500,165971,C1,C1,2.0,2026-10-03 00:39:54.290,2026-10-03 16:12:18.577,55944,27972.000000,0.128700,5603.0,1006.0
